# Path A+ — Confirmation

Reproduce all 50 confirmation fits. The verified development ZIP downloads automatically.

**Colab runtime: CPU. Choose Runtime → Run all.** Setup, the frozen inputs, auditing, and the single return ZIP are automatic. Keep this tab connected through the final download.

The original study and this revised study have separate archives. The full confirmation run has already completed; this notebook reproduces it. Results use five model-fit seeds on common cases.

[Research repository](https://github.com/mazenelnahal11/attribution-drift-selective-risk)

In [ ]:
import base64, hashlib, json, os, pathlib, shutil, subprocess, sys, time, traceback, urllib.request, zipfile, zlib
from IPython.display import display, Markdown, FileLink, Image

WORKSPACE = pathlib.Path.cwd() / "path_a_plus_runtime"
WORKSPACE.mkdir(exist_ok=True)
DEPS = WORKSPACE / ("deps_py"+str(sys.version_info.major)+str(sys.version_info.minor))
PAYLOADS = {'engine.py': '...', 'revision_data_manifest.json': '...', 'audit_v2.py': 'eNqdPP2TnDayv+9fQeVeHbCrZT8Sp+5mQuqt7eQqd7kkZcdlv5uboxjQzLDLAEaw68lm//fXHxIIhrGdc5U9ILVarVarv9T4iy+++KFIZSXhn6JxapmUhWrqNmmysnDKtfNL3GydmzPqqVM1d5qtdOSHStbZDkfIYpMV0smUU8h7WTvZrirrRqbByckbFW/kzKn2zRZwxW2aNdH9dVDtnVdvfope/vDquxe//vzq/5zF+fm6zfPlySv5vs1qCajaXbUXjkoy/KniIo0VvN7lMq6LuYPQTpyr0oEZs3UGI1R8L1OnKetke/Hub8+dXZnKXAUnX3zxxQmT5MT1poprJc37NlbbPFuZ11tVFifrutzBfA126KUQBwzQb1m1zvIOBdHpxEBwZZqYWGyrUkZHqwhUJZMszg1S4GDWiFrmETCxHgA2caMMWB0Xd2ncxOIhVkrWqpFZEaUZwBSJFHcqulbxrtLDmT3BTjZ1lvQoyiSK2yRSsH9SrOIch6ZRnCRtHSd7bj85ef3i5sebV69DzwX08aaWErfXFe6tisoi38MTEmOec9j9qCgzJeEFl1BW2LxTVdQC/rqJs6LZU5+sN/QAMpOAYERJqRBvVlRt063F9U9efvcrCcTrUNNy5rkqKza5jG4AXj8+p8dNAYjSbL2WtcTRBt0/XnePr77/PvrnP18C4l9+/vGHFz98h2sbE1HE2b0EilUS5zKFBmyPKpBBWd/3De/buGhg36PmoYxUlvY9yI5Bawd60MNzpSDgdLjiHFcCm50l0Yjln8XPEf+E+5A1W5AO1dZ1uYkbOcTqn5ykcu2obeyhfPuzEwf+1LJp68IchQB6r5997aHEM1RQyziNVvtGKs/3g638kGYbqRpPo7P0xWA670Y811PEsVitwqIK4pXybnyhn5771ItIdiDb3r0Gxz8qvA9Uu/OuxJ2UVZrtVPhr3Uq/A9BkA6o0uwf2evdCibJtcBpUDlGe3WHj1cU9rqmSi6ulLx62IC2h+vZKnl9dM7JKvA9p+jj2BT2sVv58F3rV2Xv/4ppgbhWiTfKs8jxzYL1K7Hym0T/rGt/3jf6Fd30Kw/Jy4137vrgUVzxjHYt6FZpz7Z0De+IPmQqvfNE3rlamkcY0ZYUk/CbrEljIKxJps69kuCrLnIHWZe3cO1nheMRxi51ZkWaJpFWADlSgFLzzez2BuIPeEOVwlcMhXMzEbJcV3pfipmfcssMEGFDo4rwsNhEi8IA0ofEL3KMB3aAeZR2DUu6pz2UBQuDPwdqUMBF3GrAgAan3+uVkuBzgykZ6Fjn9whDmtofJzq7EJBzJVCwUiSEqD6+OYaXZ8px+b5ckldyx0h0r7hjg2MS0DyRHnorDUK1gY7sWarr0f/dgIvgVwTPY9CGGB5lttk2IbKN5/swPMNGFB3iAI9kO5EeTJzQ1/um471b3wcDhBB0vz0Ke6xSItrlt2rUwFndhf4q6wcIaYM4V7R8frA7OnCkL/NtL39YswbNT71ad4US+uFUk46w64rZOvLxUSpAF0tsFPoasbUllszWU0jnLsorrOt4zhL+gkct5bnchetND6MHNobF1tEBVkcdNURa4Mo+4sF57yvdBjFBMlRb8pAQvAzyZEAdfcM+8zhRxLml3eOBzf4G951fLC/w9GSmpsunw8+SXgTBoQfgQmVaoWe+MsWVWXl0+KAGGRFimQyhysqKX5qHV/KtCBA+qoCkj8k48f/6S22z9bHXTsDa0JKwSV+eVP3/oZb36BmU5EMG1P0/Ch9OWBr2n4wQ+R7JlKryOKqALBCp0axQ112eu6U6ecRdXlUxDQz5uxw50B85PR8oQY41s/fMr7EpklnvvTwc9Puw5KkQPzCSA+Xrv9B48jgz/LDkw/TPv6hyY7J8mZ/Bz+vLAFZg9nHoAY22Cf9qeWa+nvCRfDA7kUQdC46M5W5rTjJ/0LA7BX45nOuZ6jJbG00z5I7Px+pLB+l6O5ptyXmYve/dlRmKn3yyJO3RrZlfn1v6z/HWujkGDLzaWITEjd4gHDRsHJHzcW+LhvZNrDX0SzEA+r5t4B26D9wDnFo9pAS699kmjnYwLfSqB7b0JLMD69VwNoc+2ezrOCnMg2qAFLyJZmGdw77dlGoZDP30Z4FGI7uO8BZUBrEtdP8ga4HBb5eS8dVb1TsDi0GpKrcelpye1XbAmrhvUG8D49PxqrtAH0WCLu7OrJbc72dqBt2/wLBokDgRe0ik6VEQTeFkfPNB66FyAIrk6R9TNFk7XtszTC+zFFhIpBVIl2EnzfZxh0PPtYCRPdhl0swE7F0T8DElehjR77woxMtyawYZ/O9y2oetg7dYR1FrNjJS0sXMQ98Liykagaxp+D4GrsXbYGpK3jU+gW9ebEONQ2PA4VR61XoA6KNbZJsAOV3vkjfwAPjgYQuBKqybG1C3INnVOjWNZiLO8BTaGi+U8KduiCS/nbQExOhi0XVzvsX0NQZesqxpOqQofn+ZJrGQ0auz8+GQrkzuviHdSlHeWLIGJBQGG2Jdm6Zrp7Sy8sjenKBsHXVpvgMCmNsDDV6Q0TR9CFGCFPHB7rUG9+Y3zPAFPQCKAqJsyD6/l+V9ErJ/+KiRozjwq4sIKM5BdYR7vVmnsVLMqZf4l6h7U0xpYTaYhyRTsNJg5WAowt84ql0czK3AXYJk7OIENRGi8HwuXf90lnOGuczDM5FTQC8gw6sOojfeVuwLQrj64nxohN0YcvblLY2CLbA2hGkdvoZELeU80RxhoRAZmICM62iMkf3J+3UpnXZe/ycJ5yxrIUSW4btIhOKeF4/fi1Y/fzx1cLJhZUEMAIusszrPfpPP31z//BEAgMs4WzBrs1EMOxKpgSGS+DocEAy1VHifSW7n/rv9duAJ+C80mOHhZ2hEfYfQKK3wcRbGrvFwNQlbSfthKUdJwOmFRYj/bZDARSIzvP9n7hbzsltJtm+PZCwbRrnfIkhiVg99Lw2Ajuj1EEieXafaF+GrYTDhxK5TDPrIDwtkACXRIkfe4dMzcobOWYaqs2/rBPkxoEsx3TQvKQJkAfjr5aO73SH03BKzQDuSpP5maa4iYn0HhzNwzGt5JOp3vMCR0cGQGoi0/JHmbovcoG29C931axjXpCzdO27yJDMIInO+83KNFiLJUmflwcagZcVna09VTYasCpBuQKs89vTi9UBLQnIKE9Ou9k/sQAWFqCDjQ6WrK0fhYRVWpsg/grSPRsC5BGUwRhTA6UFUONsS9cPtgbyWBKhmSEp3ziQwp3xCnHmK9cLkxorFBUf0GcRNYuRFQb/fg6OZxQZKkGLzfMHBTkG16DLxmq5ogI/KTQCsCcvSibDDyqvr+3rVguoC/tH/+IJyvSxAKPJ5uU4Nv6ApCg8lH2ef6LALckZkwWHmSBaI7c/VezlnwgKFn7kwrMZrPPcMfCvoAEqSOQ4tDDE5cpDhFkCnwHm5LOFxevxy0x91L0FYpulbY3jsn26xOQ7DyiwETC3d5cfHl6HwwmejoSMcCJoqVK6ApGG5Ejl4quY2wgoWLXiqcfHd5SrOegaYBQ5NAiG41KZmzt2Laelr1+IFG4I3t4gwNMu2Z6GiPYkQM5JHgIyHoYkZzL7tRbTfKigQRAYYG9OtPDZ/koCbSWWdGNa8zWQMHdccCFJE+7WGIuLM0yNDh1lg7ptLmd4NQ2UZ5vAL/E7W1ctiv+yQJ7KG7gjyWDpvhlrvsInuebxKs7cFaK/+zQm9yeLyxScFJqqLOf9aHe9650Mo+tEZGor5bH++kbcr1egDcSc8E9MMxjWHimKFSGPrfMHYNq4VhMUjrYiYul+Szj5TFW9QUdJo0Unc5TvqBqQst4nBfrVceFoZvIXySIARFKj94LkdYINAITXvfXU34yz70GOdQDnd+7c7ePr59crpdcJiJevdhq8xR8hBksXYRHDY3+OszTqReCh3uudtss5W16wta0yjrNwo8Qh2VJg+0BPjpFypIqsdh6pBrmHfCww3WZyIfRWcE8V6zypgtPyM7NfTlSdFTDKqliunUz9au2BHscGsPOK0VOiNxgI8XjxzqIf/0C3oG8iFatelGNk8T29AtfdGPXepodTD4cF8GYam1PxhbgDc4rUex56hPNVYjeoXm5k6Tf+RGz0O7GewF/VTfhsEzX+gJF+7BGHf58TkRCyitDgGEVFGFV5IpKD/UU9Dgdb0MvQx0NsK3cMMxkPFOTXCDez7qrs31MbdVy66EH9BXA7Uy13eg08pKd2pQ1ngYxh2owQNgyyMCJmFAjLajqssVRtED3cTLQTGJyW3kdR86w5wGGKxp7T4y+JPtXlFGWw31uwV56L7NicgFAyxDziMzlmMKywU3nOHPXMd4OEOric4V59ZS/4gbRAKhYYICCH3fSk97Vdg8Ul4Q9mufFwJi9scpMZUl7seOvDtb42FzbnCEY+g+Sp9x8UeTVyHdxHvMF/Saaoilx86ixRXc7HiVgaBnnRnnlLuoRrhfmguP8Og1qZn2BpS+eX7ujq6dKAMpItDGn8Tz3MZzPUZ0ZEl1/OAMb6D7VdntwvZJqFfXBsA6xz1drQAxYNzbJ1YF/et/Fp3sDzjIAjDKYLz2FqVsqHXlAJrpPtdrGGL8idFslJsy+iOcSrkyBZhvoaN0YMs+6YfQ1V9WhIOp2L0YtPSW72D4ERMPI0R3rj5q1Q8MplEdF+SjJHGr4hyMC2kQzViclJi7dmm+/lIhYk/F2ljb/g/GWB1m1CFlvc+hL4b++4uwP7JqY/gd8M4KBXu5szxzmdJCgl3cAIp0cOImloCSwF4B32gq0vtmXdOa/zNIJAyzR+3HaPJ4AsNnvD8k1kYd2BSP/+Tc5Lk+RecvHS10cwfrtvaO8fxpR1Ms7sIcihM3lDeq6gxzwnpQcCieYLJh16Ab9fnbkINcPUrLN7gKlK2HN3O8jt1mTDIXpEFs6rKtkLE8YUDvq72+8ZhmLt9loBEg6IDe2YW9REXf1nBmmnCsJnSuHxAvJ9GydxM+YqZqNlWf5Y2jmNul0JPx1T0XMAhEMLyPMWrsaXLijzB4mgGDBN1axk1bc5JlUGB1A06sZ5dZPdcNB8VWP5WF9P3jc5FPQzGcnmyJNJuJSQaINWD1zukBzNZcmQ0wYyz+Hwy3Dee5bTg/tWnDzVsgR5Z0943R5WJ6F3nnlL1nfdFJbdXJ+CefmK6vUlvShZgp4GNJeWeExCzonS2EOHl/C3ZAhkbQU3OcGM1Jc7vxbobJlvd1411ffP0V1ZokpfLe/S/FpG69XkcgWWf928r97JWaIjziMRZKeHryjmCfWX/pn5ueydX3cP7p6bX/B5UoFmU8wt8nrb+cPhviCgxdSKlq0usu7qt7fmunNsCzDJ6AV9sXklbcx7Et7gvrmsEV7H9PNq69v4fURqDXJV16hPXj4f1mp+0sR2zgK+iBw8ZpFNOWT8fzJlxCWM+8DP0n/89dR+fv+J8K9cnGxHeSS4NI5Ptw4tsQw6yFS0Y8o7ANzcGyuztOykqCjcH7VwiPt9kaTIer76mrALSZ6rEdFNaN/+y5nMauRVggact5VR9W3+guMtjhZHqBISanwvolLtzTFWb7MLwC/lX1NxjPPwvEoOuSujjWvwrEZQAhM0T4smp45m/CQZJiXuhO/qHz+UdEtMuxEH8/mW+JOER8MjKipbirA5OmEoxSKdhMxTpaZDWptAJ+lmkvzHoF+hAgjCmvsiSeJ1kwrDkxdOvMTZ1AFHAeaF2cMriXREhEZVr+MZMM0tfKCOTQJa0McnDcQAIUS4oZcygwEOWkmfb1+zos3Hfx7BRGiKtz+Pe4HrZjKAuXg4vgaJkKLrD4/4Nv7YQFa28HEWr1RQj8cSe4c9XQz+v8NJNcnWZOKtcZeBshayNUIvxUbWMlIZzPwEiD04OXQBUWctnFqfafuE3CQeG7pzEzJtAFNFyYVkvPYzrChrUzGNe2iExOXK64XCzsMkisCvvXA2XYd/Xq0G41LjL/oqZkRfxHT6o+mU6H2bl58+rnF90xTIQhPkhrUJG4xeV4j7t8l6nDOFB48Gjqxe3KDpfuSvjzAQqXMTk1SoOJOOScFZeWgZzVe/QMxcq0W0iWR1MGVQyGMnUI9hxh9a5w2gaTBmhDwMmIqPDDi8nXECv2R1hfHEDcMMTNcYjnDPF8EBXXwTrLwbKBud3ID6H7H4rVfv8Ph8a//6f6H9e+yhLqj8EPedyn0SDYAe3HzsLsU4xaYfVKTLf24LOpA/5o3oNGjesML3Vt7EvwNZl/Zosoo2FwsrXtocZp5gqTc3Q7fgG6Aa+8g3Z16+Jq6A3W8mEDthgOK589DVQ1VjbWLkhawNKW4aPbYzc1FTOsLeibfVAhXS5v3wGNi0hcN8AEotcJ/vAziCcrJTwqjWJKjuHTydmOMf2xOeu6wK3L7GsmU+PqvnGtY8bpZqs2gcb7o881LDFBN2goFO/CAjPs6ILxaCt1bwjEEkCQlk9BmgTqOMU73s1D/U8fMelu8/HSu789f5FDUEbZ54MRu3DQ7/nzHSXHI5rM3us5lfHsgfxdAOYjzZLGe4eV7RikMMmjT0z6pMmb5393aO2KMiEgCOh6gGaVag5xLAhF2ibS9DmN/ipt71AFA977wpnEIhwVTK/ZfLhVEbJEKqqW0hx43eAHZXX6GsuUD1nwr3AIAEKCktqls0wZA5dTRO+iOn7AoMtIEpWyfXntHi4cQoGsGTEY/Gowkjus81ecgID9iogVoAAgosUCmmbbd9ErdNECYVURpjfC4PKZ+Gh03mVu2hWGxzkM+YtIypxfsGILpsQ20JIRh7EhOLzl6pb9tdBdZWg8Znm5IbPuft58EoRZ3/iEoMg26DG6AieL+vs+9BuK6BbvfGid6E/HWD6Aqe603NFVu6R8k1W7E7n+4nwyLCdG4655/xLD3dpPBdqD27mmTEDBUD4ARM/ImSWVWlrpQJhsOc330VMg+KyPiEX9e3hstZzSp5eHaV2+gMc+vrbqzyQmeiM0FnT35iZV6wr+NIUvFY6cx53GVhTBazBPeDMFJqpr+zGjNb7r8iDi66980Q/JfnzjWe8a/OuvPhfsy+tPg315jd/9GF1E0hARr4kb7NxwC94z7QKUuonACyvUHY27jDY1cO+I19wrN82IRhZgC2BvOZblRn3UwS7gdy8AArri4/ULEDl029XLlPnYlgWmuwPTNByKjl1TbLzGR6p8c2cwi2BTrTBMaYvmnAvbhClfO7yqnn3sGvupn5essOfeYP01foCJc63zVm0tyUpBWdQKr+f19S3aUu1GdZc6+n1wq0NxDh6qDoq/OB6nYGFSFS7uuPAeYW0PgarnOR2tcLc9PdOZe+GeET588ic8XdoSRTeqj0OfA0zwoeuz7OdHgp78+eAa8wieCffoEBG6PRrBhAd0AD8haROLRrV1jpWU/Qa5wl54OKSf5qddwEFdpR7TZr5G7Mv5NcpeGDUJ7swUX/JuujMuwsTLCVCsSRNZNNiHQ1kgwwvi2eDVxoTMIqmaEZlPpuxctXkD/qsuEp+5uszCVMebOnjtDeMrf41sHyKh2yPTakYJXXE6sw+loAq4ntOz7lGMSstnk4Xogr426KrVZ6ayuWtZCkq9WdyLWF0AYdDOPjThDaoYU5oX/ILZeay9tK4C6WsKXRviBw81HG2uDqGCkhTUmfKYh/hZKl7rXPtnfeU46wQL9vFudq+/iqGPZ3msuayjA0psp5JU5t2YXf5TP9Xg20fGdXICSKII1xJF4ABHYGnBtkfaA6b/kqAOzX9OENzUmxbTrb9QO7i0DBDEKag53ee5yB73SB//bwqu4AANv5rEpCMmcKiKZKNCPYx+cKDyBsLHX6xge0CfrdAT4jRfMGcgea/3Cnj03QeAvKQ8K40dfNxgJJckFQ7h/wOc5Pqy', 'audit_calibration.py': 'eNqtV21v2zYQ/u5foeULqUSRk7QrVhsC1mLtl6FF0QxIAE8QaImyGUukRlJOvCD/fXekXmzH7aelLyGPx+Px7rmHp7Ozs+88V9JY3eY2MKrVOQ+WSlmQsCbQ3LC6qbgJmCwCwyueW6Hk5aOQhXoMjGVWGCtyE5Ra1YFmjwHTmu1MfHZ2NhF1o7QNwH7eas2ljcvWtmCzX1kzs67Esp8+GCX7cc3suh8rM3HmG5CBetCJv+2pmN1gVLZ1swuYCWTTixrwHgTwtym8KZOLZhej/6Y3tzHZDd42emTGcG0sFzIr4HpM5nwy+frhy6fbhBKQsJXmvIb7kIg8mEzJagcjzeSmH1ei5plUwnCYgKJWDYpr02QtWNOWCWl3bo3rlRs8NRBbXmS5MmhXyKa1w/EgMEKuKp59GIcf3XAlYVMhypJDhJ2m3/rn7TD8/vlz9uXLHyScTApeBqwthAU/hKWWmU04mwTwg/MoL1cJyuY4SzDAFEeh09C8TGQTV4oVTjolIPGnZuB+xSRDcJhYNv+ScJ6z6kgdJGKpndKpDQi7ox0DErMRal7beWTXQhcJOL04sC1JOp2+cRpw9aRi9bJgwf0MbJt/tKU303dvw3OYQazp/e94yILossweSXoxzpYkHS7eXRPECZikIFmQe1iPa84kvfJ6GNzRTwo1InJuuvjiz5ZVLTfJ4hTCvE0D92CapItZ9JBGMNkTdfZgIQxKpYOHQEgoObni9H2YDofgUskZFhoqUIKQAax8VZLv+eIuluChnXIaiHLYyCvDA+fRB5JeusFHks63CbrUb+g9er2TOs9xqxvA1nBQPvDARyRmTcNlQSElLp4/CJAPytb9ehUCHZs1a/jiOg3TMJycPqBmT7Sv8yGJe7G+PwxzPGTz1Wnd3vHQH56JqGtrShE3x4eMALo8AFl4fn6zZ1BzCK0EQouZcfxK/RFeI1/zfGOSq3nJRIX0mix8kNHnO/TZVYjnbEPSEQMN09YApw1lRiKnWiuoPqUzw3lB0uhu9AQlCXAX7agb73/z6zuK1B0XwLyGOqORAUrNNnxnkr90yyPDQczAJh4XwZ8ZCcMYrqsKTmG05k+FWHFjabiY/ZZG1+/2bi9XSAsQ+ELVMRQZayubgZSiN+EB8DVvKpEzy8dUuRuNRALkcFQFy0rlm6QWslPFaVZxubJruLzjmPBgA6AC4wYexBAKvgKk0iuveOl2X1xH+H7FORcVvZs6WXhoowNA4lhIosdAhpwuMMfeb4jhhd/pbmbwRv5kKKbZXXrhDjww2j8iyQkWmrPctkDJjt9Kcvd89wI4HOJ1WJYeUhcJRIG6h+/I+TKQyjo8VlVeKcNpf3Tkj4m0VVVywy/fR6wbXV8dxd3lrINsXyzlmKkpujgdHHweRi8d+3eATpoi1pzBy2m2Jx6aTiuGVRKV8LbYrNE8FwYWE6JVK4vMatF0RjHU9C4CX8JoBasNhr234QTLHe2LyT3gxQGg9vLa5RE0Lr3Bi5vzw5z9LF+DUpc3d3ZsuM1Agz9RUnO7VgUJ4b3MFxXs7jKVjqwVW5W5doiO5n6S2v8nra9TOvSNXTJ8ZmHtGf712ew47plgCskM/we2yxplxBMNI+L9JjP/OyL9MWTWj1722huqAUadYzj0vYwTzrHJcXyFjYahTgqYUbIUqxgXIKYOUJY/AR91rQZ0RUCsFKBJ2xBp0pdl68oSyI4XvSV03YCNVaWWlJxPz6fIU9k5MF7aXdUAgw00/Sjs+kSTHH/TCoBgvilVfXrieQvsiQ9Y9qj0BijHMdbbCJ+064gqQHjTQvfYAjmHAXh2E06n8IiEru0FG7MjokQf0Hlci2vW0LEvjNxtj5sFt2Ph85N2AT2QYdgqgN6WZ1YdBWNM5QmjA1b89FCl0fjekNvjb5Mt16IU8EBFh15Akbdm7Z6dHlnY3yfPBOuiBcCQBtuLgiDgAekUQa8XI6JSHyH3hHj/Qt/WOBU8cUAjvuywtZue2Bh1EZh1AmjZc9VwcOITXGB36pOrf8Hwkwvc61V+8vU1R73rt9CAWlCBRzZCwZuBt8iLC4RLSQwPMaBs6ieS1fyCZPuE6WDQ1UEYP2phua+EvTfehzRCJpI2uQkvyN+yq2Ofrj3d581s6+KyibY+Mrg3BrO1AaRCDja/JF2UXkaTB6zg90wmoJxl6HOWJQnJshq+o7KMzPbz7OsfPgiBgFdbbM3mmsGXWHC7g4ay/vQEq1d4rNdf9LBIwWIPDJft63DyHyXoCRQ=', 'analyze_v2.py': 'eNq1W3tz27ay/z+fAjfpLciYoi03zkMuO5Pm0dO5SZupc247o6Ph0CIksaFIlqD8aI6/+/3tAuBDkh3nzNxMa4nAYrlY7Huhhw8f/qQKVSeNEuuk2Oh5nVWNaJLzXGmRFKloVkrU6q+N0o3C02U5qpJC5WKRLTe1Eou6XItkk2Y0W26aeblWOnz48OGDbF2VdSOSelkltVbueZXoVZ6du8c/dVk8YCRV0tCEsBMf8OiAis26uhaJFkXlhkBEigH8V6VubJ00VV42hL37Gm608uTL5VL6u3BhdU3fGE3ePHjwy8v3b86iz1KrRE7k2ZuXMpCr60rVVY5NY+gf3UMgk3STNzGIU3U2x+RLesa4ytW8wVDWXGP0Te8pkFnRgNvzJisLzP3ce7p58P7X12/e0euvludlqRsA/PHTj1i0zit8f//uAwG9/PmXyJPqqgJalcZzAgxkkWQXKt4Uep7kKsUAjcdVrbSqLzDgP3j95uObVx+xNM10sqyVWquCVupsWQBPmi0WqlbFXPFYscxV/LL7SmQAvi4rs4tq08T/c9Z+/e3t2/j9+9d4zatf3/36G21iSOFEPjrmf7u0Ymp+/OTZd+e7VGNq/ORZ+mKMjb97+aPhzhbiWn4z/2Yf1l9oYB/OV8THWiVplCfr8zQR1aRKQxqI5/rCq4JFXia8ZJ5pHE0k63JTpDFOscIOH6RqIdZ4JCXx5mW+WRc6uEhy6Ig/eSDwr1bNpi6E/Fchwz/LrPCm8t/yAP+bJ7vGp5FgMDOVo9FIzh7nqoNisNnBEMU6qTzd1EFdXhoAsShrgSeRFcLQMrOkzjfYt5eXWgd6XtbK0ljWqaqjogqhoRpq4fFk8Ckr0khq3pz0T3XEw1OGnp3mEeGxT4xGFakmLHU8xd9FnjRFWfyt6tLDIwmVRzsYB7Qj7c/67KGlh2Y8APB8s9abtZf7U5oYjWeH9Gn2kBRJfv238uqybAKYGcdn+xiRteBJP+CvBHKKP+H6U5rVHiwQhFdHH+uNCtRVBoEoP/GTz3jYgEVki0Icfao9xhWaZYfmoUjW6kDGYI+qQBgmYl4W0jLp+0aAGnXVeL7FqiFyjUE+JZY2Gy1nUSQrmkkl21c7u9jkeTxfqfmnCsfbxLWClbmWhl+Q6OwcNhqyGH8FqTvLvkjqzop9ZPOCKslqlUaEiN95KM0IltVzaCZMRaIbHUKhSIrYJJIq9Ra0g3GS5xYwVc0uYDvYARoa6myd1NeReTPYMZ96nn2A1FYKNBNfr2O9yhYNWUHxrWhB0qRJtGr+KxoYZQJxOBoQsaYP/J/Ujb7MIFqwJxdJncFjAvbfYgtbtIvNG6ADAAhN8vicbArIN8ThVBjbLrq+NxmgszsET7B4FuLx2hucJqmW5ZEfRSdHLG92IFzCplXn17A49lXkZ8pU5XLmhzqDrvmh+ss78UO8wBswPGxKNpTQr0PJhmIca4Wzh42FBzSHHpCiXEVvk1xbJTOQ0dTINJkrLw3WfsCUkNm6B2kBWSqL1JgAYwYu4Rcc8CR14JM1/BexSU4Yp+FZmJGkHM0wxyfSTprj7maxJS0nxEUG8G/aFxLxa9WsypToJpfcEWMJmpp5qOFaJQX0h3HY0VlIg55/OgDU6Q6YblIvTctFNPbvePl0PJkNCcDum2SIa2R2OfCetxPNCBzp/LCXZAPGhBugfQTvIM/LS1VjCTTF83jd96OxGo2PIXnwAP7WS1bZcrUF/8Md4E2mtAMm93auzRr/+2iwqqXPCGaYVGTXPfKofXlFaPAakvW2hlH1zJh/apfsUQTiktEGnd6iCLBnUWvTjNnqHi8BDnGOxkdH/rfd8Jdt0awj+pjQM2L6NIwJEcoUngkCYS1u1zF88gKyA8ly6dGGKGqsYaXIupdzBoFg+IFOd2YgADRBurMzN4fJa4z/wRvZlV55PWYf7zL0+D4MXWRX8EWtPzEs7R4HLO2G+3whLfJJK+ulcjZzuo83RIhsLYczL7NZAJf1FfDBCgTBUyD1kr09DPbPI3GzArNWZZ7ex8SaJRBv9osG5T1OOpC1usjUZXy+SZc4GF3CiSsnAJAx+LOliq0kuOdODFoIlofevBGGZD5XlTM5Dos9B4TsPFpn+lOHcLiCse6H5xfsk6cBJ+5mq4XZz9FHopejiVbdBEc4ZW5CuByf5zmHTfrQSJtGkLtOYJ+TiyTLSZKFVgjSsDa/Dk1UV17EFs0ghLk7oPh2J5xoiYpvAdwfCs22iRhwqYcq7l7gIPfzCnxoZa9TsxQ5Y5xuqjybA4X27q8lgTS8HJiku8T5PutJojlGtXJo41VOop300RALXX/SyHKDnIHYgNgYm0MY18e1d7LFu09MeywbsJ/Gk3qtYytNcStft2l+Vd/Tp3jbTiXidI8prhFifo1vSHIQGS+coFg2IHGOyzrukMZDuB6nh+uZ5/dYbX0MU08salZxUlg4OjV8Jyz8TOCb9S1GoqoHXN955922wWqDRkh7mmbJsoCgZHPdj3A3RdZQlEZhq0o9m9DQqJaQ5bw89+Tjw8eHbNsfg8oujENszG40ImgQT+cP69eUW0gqigZOGZJCHvoSamhb48lY+tPxrAt01kmRLZQepo+E59CqTuwgbKa4myi6jRn4AOKQ0P7adVmj1trzd8JhbRI6ftlCfjbLb2KasanfdcQPrAdc8EOsWZmxqjc2QEyUVCuIaNAknxQRMmUBkgHXNZIC3pzid8Li+98TtVMJJYJIz/wAoKza+2B/iAww0thCszrHCA6xaiayhRggEpkWRdmIX8pCCQXpICIoKyPEGNMd2iBtrmGLz8syR8o2ZBIzyuyX1Jc2NDutOMkeMOCHKDzZXajmZZ3engSRfZ3Q3y7psScomX9yYtgoC5Pw1H4gX5vwf1KH/YqhTQPMNKR+HwANE8gygfrEzWUZ6yyFgDt82+O/wzC1eM1kCtc1t1Z0uKw/M1jYn0hMqNDajIl32/IfjvwWw3mdIR6b4Nh4BGuqUR1e+48fH2MWQckG+K47AD6diCAoti0W2AzVT3k+ucrWSDOAIhgDSeXecrNzdpAn4zkgVJJkuN4VjO6Mpyy0qs5gI9c4uFm0Ra5DxVTfiYghS2wpV8ZTMa4iKfauAic3SU4Ql8jGFL3t+/AkOHncvZJ32r5/L5auXxBfHLtaP738aymlD8/QFJiP3Rf2jHKb3THGDpSYj/BM5XFZ5Neyb8uGp9D5hp4l63uMnimbr8iapH3A7h19QFLyaGh+7Fq2QJaQaZ/C2a5NGtDpnJLb8P1MQuf3doTc62YHdshYKLJGfhT1YK53YEg7zK5is5E9L7EAt7/BAexFT4qHU4ZKV3V5npxncH9UBZgwb+vkOlZ/QUIGWwnaV96uvQP48aj/6O/Q4RTc1sr6ZYOeIPqDiIOtbtyX051Q43SAyR3vFhpTezXcbSVgf9QyQOeqz1DHFAaz1lRlnPlbCVNjcs8W4hbMlJOxdLsIyLQYptxOGzTStntnw77YsIPWBVNc72Kck2ExjMa9tmtm+mXbIUhkSzYcGtvvbYqVIiy2Y6whUbRGHGwrgUM8bo9Ov6bcM5yms8C076br2exgihCnnn46eMi+6+FsEj5Z3AjPjunUjvima/PJ1fFmbX3I1rv7zBxyYDoZjWdfz4ZlZAstHRuO97DhuM+GLmp20b+/VWvsk/tFviz53Z+Ch4413y1uenwwFSrLiVqRf4AEPeJ+sHh5ICCC2SJTqWCVrdeskW3nGbuWgZBvIakCcouYe8Q7EVyQEt54HIjjF4F48iwQz/B9fDT2BSf26zU+iBOBUFckpQYxRLbZ1Ocm9+JsH3BZQ702pZN1lSsdilfgN0WBJ7yNRCxIJ0Shlhyy86rxYAouPKOpULyjUqigFFO8/OdvrwjLuQLddSjeQI1hxIRelZeCmCU880bejDh77Ye83bYVOZU/woKtkLFQaeS9zdje2IKvsHkst0VFr1NM1I+6/ugsaGXcJ/Ty44oi664IQk3GjQZhdDWAexmil/OOXF9DIIEKxa8Aqp1Twvs4F6DlgssRwpYjRHlOr7cpruj1zLtX4Yyweaq9rOqyKPNySRYf4lfl5TVFnjYpCQVRDMGYr1Q6oLwtZIzIlbY1HPBcbyqkTADvFWmsLD16JD5y/eZ4IkwiTSf16yuRNOJ3Sqnvcwhnc0ooxWv6yv128XJELXXqsNPnm7a1zk11aqXPgqFimcP4B9fCh4T0ZOZjGxqMqAqXQxmzYilcTSqpkaFczfMNom7DpjvYQ2LbP0wtVgkkp/W1glv/A6Wgd7GTNL5MnHEhceTainSBxNXdzLrOowrOuAVBGTphC+jUESvg3JHSE25zL8WczEL+3LViTUg5EZ+NQ3vInVQN6xLcCNO2FGYoAF7aP2FT4BAkEJuys9x6Zbp6XV5wolbGlog/fvqRbKtgm6J7EvKBHbAzQ4IL3WyMOtutNxDxJmAfbTJV1+8ie+i+W1tsqtL38JxISAP5a50tM+gec1f1te3cySLUhC56aDJ2CzqCgcZJJMN3ULNVg8Qb+0XRvoz0EEG4c7omBJFpqcCLtp3WvRpZhndtJ8s83tXKYr/hXNFCjsRn5vvNqfhskNxAVva1nm4EN6cCN/vDcNK0omj2trbSjWjIjpAxh4D03dgB/BiJrVGKfq+d85wC1mvDHAVpu434gUgnqTO3VqydkCfzutTaSjbigAZIk4oIMOVXo3SGgLZEO7LWpNXc0Iqtif9+e3P2z3cfz8I1HH94WWeNMhWh7naL2Z9/QCMBzrok9YrkplmMnre9etgjpJZzFX2WVMCKjR2hK1I2CK03RTc4o+tGEGoV61VyfPK0hRqOAswkk4YNLZR9xPSgPw0baOobri0edF0mkgqaBQdcm6ztHsu3rDKDkGMOzYA54XaCMQiHXHY77IUPhy5YgLg24Ik+5WaBi1boIkyC3aShvNniElSfT2lwicPtKtpzT8PN7eBpyibhpKAlvcOztfbgS3htcZou4+hMx91rXL2wJx5cY0yRIGmvg+OkoWiiYyMsrldDVwl1d7nHXTYo9ghZsMg3emXv7vDdoJ3lE4c1zerIpjAMIk/NoL0VtOciUJU3YT3/gBBgrcNNBfunkEFX6SJcwKhS8U5OnhwHstJbA/QULpJ1ltOdv9fqz+R/N+IMzljaObpSIScvwhNI7BV8mIZjxkdTVnLCOdRwHBZ91Qxn2IL10FxBUz/1R5/T6PXe0VwtYQmZ5nb0puV+RNs2PPLwQRCR9xTrvguf+3yPCmkcJkLYnXhZZ6mG4fXGwXFwmaXNKmaJQaIyDsbh+MksuNRVAkU/Cp8dmXfkatFEjAb5FOzkeYvkOBgHKwf+9Il/yhu3sOO7YRk1zVIJJc8KnvLCoxPAjV+M/VO++Rabkk+bR/H1VTx5Q7dKyVIg+62u2tTcoQrLGk43ycm57SSuXbrVrW0tq+uycXYTeNadBmwtbP/CJ9+nGB8Jm6Gul7UlVy3nwQu6reoRN6efZv69r6NQN4mCeseCQSFfcZDMqREyIyRGyIsoLdotQlIlK+o3Hw7thg6N+VtwUSn+TH9v5Gn9xZI/ldH/s17E1MDM9lXQTd6+v3w/mVFPYVBlosbDsOtAVxy7Oqd3HUVj6qZ+H574wUkYDGaooVZxiQqyHxyFUBfIAadQUe/iZY3Ml9MqKjnHLgaRs95rd8uXj8RZo6rRYlOYQItjsKo0QmkTHM3hhrnbYzJIWPoLcqLg6g5GczE0IkrotuY6K7jMha8INOv5yrapaAekVwGV5yNJ8gbZ574AVvqjsb/LeCdjLuiy92F34AY6iRCu2S5SWh1pS5WsK4Nypa1CTMxn0F04mPDFXW8Ox833A+zzBYIyEvV5cEGS/ndWebw7S+KQRi4Xck1W81fPbYzENSkiHjSlviP/VKd2gO49HQX7bj4lVyFrLb+S1gVz5K51ZC5Ku5jW5AWRueTcDV5G4/DpiW05YYjuAw6uP5uG05iMHoyaHI3kF4HlSO5QuMiwSySRl4pvu4HSXhX0iMke6dTnLwc63b+FJK9WSRSOj4juI79nwqiG5F3l2Tryjtg++8G1eQqoa0ZPvH350l78sMWKKzv8qr1SguAwh3OSXuIL3uon6KDdl/QPbNHJyNDsQB7KA1d+IjmaDQr/tHSyzQnjKaG1cwSwEOWanRfidqqXloWpdwaIcfPou8Be0YbfoYA3fBGsEF/nsP7Fslnh6L4bMIH46jkmnRCTYDYe7LOsh7106xCeyfSHx+OhUd17kbRvX23a319iA1J3r8ABRu6Z64Lu4Qs3Vr9tAYcXDNxoK4WDXxz4M/jlplwsBnR1l4G7CkHvqu8eL8iBAsIJ/9Rp2GWIkwsuwzaPObwMW2xWZOWjk2dP1PMX0ujWmFcnVyuEDwoRjQN6xv+k0yk+q+dWR2v5zV/x799Iu/TCLD0+OmpXv+B/ZvXELnb76KtCcPwEDHPCD7z/es33KT83we83h/ySVgXO+CyEqQA6PZDeud+/oHT4/t2HwLh0CAsTeC95bom7XUSTq+NbjmA88///nSocOcK+Xin8tninPfDISpkRafcwlFQ36iTVfPYmBvfiIvDEn3UitdMgcNfWIu8e/v77qEXkw9lQKuGxv+psho0KIVNVGp4pqgZzOPHYvQhBUZlD/pYeXhPAnccVdYYR7NL+TEHiFAlXiaktPF+DortorZAg/o0NtpQd9rCH/KMFhOBHgekj963fsdHRrbsW/NMQh/VrHeN3X3aMpLT/iVtkceuiE6t1kwzU7oQfln/x4NKinFx0M12A0ucWS082OyS+y1YW5KT9aqKWzEQtXYrg2OW3ivkVVuiUwPeZoK/wxkNTZFDebjjIaFiDoeMk/XOjTQYThUfPTwKT7oUvXgTnZdOUaw4fkBRj6ElvfXKh8OmZ9N3l8+MQibkM0iqLnmITp3dCFsseJOW7c2gSp7t72qKDILXthjqcnKPFib0m+6WOLYvSLTjOb8Vhyi1bu4hNEffuYstnR9/g54rkGbj4xz9SPPzjpx8nXJsdsbPg7hLZK1MfNF2ms9fBVpNmUPJyQs0ZuKCwTjSlGJsKlylE26Ia1lNBn27LZPOukWZqFFREt8wAwR8g+lBN0+GzfizoNyl4HxyODBpNgSBDa36wWicZaV1rjU13rVwIguhXvamDY2uh46P/FsbW9zoUp9RsNK0O0i7qs/CVL9IyIntV1tnfZUHlNSvgsQ0F5YRqH4G8gJVBsEViOuHfnkmIYEyyOHlKWo+JeFhH6uKgchGrK2t35Nusxh6qPncaRZdnwAcLxew67c5XWGus7RAxjrMgkm3X5zJpJTdaFtxB4D5g0djmZ6L5fDh2hjoQawveHgvpzU417wGMbBzTz8/iGGY2jukKdBxL456ryP0YOHxZLzcUDX6gp5pu+nFgkdhhj0vEcne43NBoElUh46EZjdXu54FJyEXAJOQq4v8Bk+uA6w==', 'portable_runner.py': 'eNqFVstu4zYU3esrCG8oYTR0YrRF4cIFBpnMqo8ASTcNAoKWrmx2KFEgKSdOkH/v5UOyE8cYbWSR574fx7PZ7Mpoaz/3SrhGm5Zc3fxDlBi6aguG4AlxWyDrvYPPvQELZgc1sZWEzslGVgS6jeyAZdkdwkQteodiPZhWOhtEvb7HLYAidmga+UTyT1U/FEQ34dqKFkg/rBXqutnfaVNtiQEFwqLS6x2YPdFGog2hSGMAnmFe6a6Rm8EIJ3U3Rz+r73ZoSfhBpEVxJ9CnmpFvg1JEdtYJpaDOUJtFEdIioBZORHClTY0xrffBnxQPuTG6AmtJpYfOeaDoiMa4glH0xYJzstuwbDabZbLttXHE66yUsBbseBRfSq7ZaHS8+c/qbvytbdYY3ZJeuC1ikxS5wc8RYveTzmfZN1LB+Bk9zrIaGlL5WvKxljwmLPdqy6rZlBHKt8Jui2VG8MHUuMGuUtSYINnsucAiyF2UKwLMx+SdCOh7iu8N0IfVitawA6X7FtuBYorqYwi+A6bSba/AAQ2qHqXbjiGwf2X/Dd/RElohyXR0zj+Yl2foVj5bTGlR2zxBmAFR5zSGyFvRyQasYx5Ii2KST55HNfcUG4E7qLadrITiG+EA0+UxNX04J5OyZrdi8fMvPqKjNJ6TwaY6lWC13KCPefq6Cn2cF8yD86J4D8OKjVfnzFg31HseB+KsuaO2ZMLWsnrnAQta0MrBjF7HSV+d9i9LU5RT54eVnvg2yjLbK+ly+okW9xfo1OhzEONJC314D6ML9iu7oOci9i5M1T4N2cBOer0TJC/uL08qazEmqPNDsWKD1nzocG3RByzFCKEv9ev8pX2dW8D7y0sadmKNW2VcFV+/3H25vb67DRft0cWff3+9/uP2JD9nWzmWcezgKSScZtn9sIhT+dJWCFk+nt1Op0NWDbVg0nKxE1KJNY5fUdJb3LmVmza2wa0ncTOPBBBTFa5Ct7BYIFy1g+nIC7V6MBVwnGve4UanS7+74lgzf1AeI1LVlikAvwnCMEV8mWaaLqPRckqgf2jyjCeuocsX6hXQJfVx+3A4QjowrN/TEg+RWfZ4+xHRhFYb6abE0lVqqHGrH5HWepCqTtT1WziHJ6gGzwL0rWcfPHTiHh67fhyiqf2X47SUaS7G8CZArBkfDzj/sdUjnuKP2nxHSbr02yR9YDEmAudpmXnCF90G9+Dyzgzw+hoJpfW9d+CKDayQiJgwmx2OFZENUdDl41Hx+yUBZYHEVjZtcCJ2Cppffbh1vJKoefVWLKp61+Sxxc9IHFjDWzueGAM4RRX4lVqmJKxa8ZRfli3G91OZa8vwXwkPbI8+Ydcvivl8cbwTR44USqJqSOS6+pBy4+DF6S7fciwWOLixxpZbhSnhPEwAL1gvDJZl7nvc0vKIW/3YHDK/+CDzi5iuv3QH0eX8jG7MiOP4T25QI1myRyMxHgdPLg/LqR7a3uYvNAVAl9aZo11ltdp5UipHkl++J/2Sytr3l9sf7qaTh9dSdv5jtUjZTVvkIhX27P+HGOIiyxDHw57hHO85913Keaq/ERJRt3vroL1+QmqJPVxk/wO8XrkH'}
for name, payload in PAYLOADS.items():
    (WORKSPACE/name).write_bytes(zlib.decompress(base64.b64decode(payload)))
assert hashlib.sha256((WORKSPACE/"engine.py").read_bytes()).hexdigest()=="53a0d22652fdff2683dc9053f8ad072f09958b79b518f0ba5edb63ccebcbf4e4"
ENV=os.environ.copy()
ENV.update(PYTHONPATH=str(DEPS)+os.pathsep+str(WORKSPACE),PYTHONNOUSERSITE="1",PYTHONUNBUFFERED="1",
           OMP_NUM_THREADS="2",MKL_NUM_THREADS="2",OPENBLAS_NUM_THREADS="2",MPLCONFIGDIR=str(WORKSPACE/"matplotlib_cache"),NUMBA_CACHE_DIR=str(WORKSPACE/"numba_cache"))
SETUP_ERROR=None; RUN_ERROR=None; RESULT_ZIP=None

def run_command(args,log_name):
    with open(WORKSPACE/log_name,"w",encoding="utf-8") as log:
        process=subprocess.Popen(args,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors="replace",bufsize=1)
        for line in process.stdout:
            log.write(line);log.flush()
            print(line.rstrip(),flush=True)
        result=process.wait()
    if result:
        raise RuntimeError("Subprocess exit "+str(result)+"; details in "+log_name)

def fetch(name,expected_sha):
    destination=WORKSPACE/name
    if not destination.exists() or hashlib.sha256(destination.read_bytes()).hexdigest()!=expected_sha:
        print("Downloading "+name,flush=True)
        request=urllib.request.Request("https://github.com/mazenelnahal11/attribution-drift-selective-risk/releases/download/protocol-v2.0/"+name,headers={"User-Agent":"PathAPlusResearch/2.0"})
        with urllib.request.urlopen(request,timeout=180) as response,open(destination.with_suffix(".partial"),"wb") as output:
            shutil.copyfileobj(response,output)
        destination.with_suffix(".partial").replace(destination)
    assert hashlib.sha256(destination.read_bytes()).hexdigest()==expected_sha
    return destination

def safe_extract(archive,folder):
    folder=pathlib.Path(folder);folder.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            target=(folder/name).resolve()
            assert target.is_relative_to(folder.resolve()),name
        z.extractall(folder)

try:
    requirements="numpy==2.2.6 scipy==1.15.3 pandas==2.2.3 scikit-learn==1.6.1 matplotlib==3.10.1 xgboost==3.0.5 shap==0.49.1 river==0.22.0"
    fingerprint=hashlib.sha256((requirements+"torch==2.8.0 CPU").encode()).hexdigest()
    marker=DEPS/"path_a_plus_setup.json"
    if not marker.exists() or json.loads(marker.read_text()).get("fingerprint")!=fingerprint:
        print("Installing the pinned scientific packages and CPU PyTorch 2.8.0.",flush=True)
        run_command([sys.executable,"-m","pip","install","--disable-pip-version-check","--no-input","--upgrade","--target",str(DEPS),"torch==2.8.0","--index-url","https://download.pytorch.org/whl/cpu"],"torch_setup.log")
        run_command([sys.executable,"-m","pip","install","--disable-pip-version-check","--no-input","--upgrade","--target",str(DEPS)]+requirements.split(),"scientific_setup.log")
        marker.write_text(json.dumps({"fingerprint":fingerprint}))
    run_command([sys.executable,"-s","-c","import torch,numpy,scipy,pandas,sklearn,xgboost,shap,river; assert torch.__version__.split('+')[0]=='2.8.0'; assert not torch.cuda.is_available(); print('CPU runtime ready; PyTorch',torch.__version__)"],"runtime_probe.log")
except Exception:
    SETUP_ERROR=traceback.format_exc()
    (WORKSPACE/"setup_failure.txt").write_text(SETUP_ERROR)
    print(SETUP_ERROR,flush=True)

## Execute and verify

All effect directions are retained. The CPU launcher records the installed PyTorch build and permits the CPU wheel suffix of public release 2.8.0. Operational process count adapts to the available CPU cores for confirmation. Scientific settings remain fixed.

In [ ]:
STAGE="confirmation"
if SETUP_ERROR is None:
    try:
        command=[sys.executable,"-u","-s",str(WORKSPACE/"portable_runner.py"),STAGE]
        if STAGE=="confirmation":
            freeze=fetch("RETURN_development_40197585b5fa.zip","db71d50454e2780cc8a13d9fd4750e80b49a103218b02f82ba9aefa265e75592")
            command.append(str(freeze))
        run_command(command,"experiment.log")
        receipt=json.loads((WORKSPACE/"last_result.json").read_text())
        RESULT_ZIP=pathlib.Path(receipt["archive"])
        root=RESULT_ZIP.parent/RESULT_ZIP.stem.removeprefix("RETURN_")
        run_command([sys.executable,"-u","-s",str(WORKSPACE/"audit_v2.py"),str(root),"--full"],"independent_audit.log")
        run_command([sys.executable,"-u","-s",str(WORKSPACE/"audit_calibration.py"),str(root)],"calibration_audit.log")
        audit_path=root.parent/(root.name+"_independent_audit.json")
        shutil.copy2(audit_path,root/"independent_audit.json")
        shutil.copy2(root.parent/(root.name+"_calibration_audit.json"),root/"calibration_audit.json")
        # Append audit evidence and rebuild the single return archive.
        run_command([sys.executable,"-s","-c","import engine; engine.package("+repr(str(root))+")"],"export.log")
        print("Verified run:",receipt["identity"],flush=True)
    except Exception:
        RUN_ERROR=traceback.format_exc()
        (WORKSPACE/"run_failure.txt").write_text(RUN_ERROR)
        if RESULT_ZIP is None and (WORKSPACE/"last_result.json").exists():
            saved=pathlib.Path(json.loads((WORKSPACE/"last_result.json").read_text())["archive"])
            if saved.exists() and saved.name.startswith("RETURN_"+STAGE+"_"):
                RESULT_ZIP=saved
        print(RUN_ERROR,flush=True)

## Return the results

The final cell exports one ZIP, including diagnostics if setup or verification needs review.

In [ ]:
if SETUP_ERROR or RUN_ERROR or RESULT_ZIP is None:
    diagnostic=WORKSPACE/("diagnostics_"+time.strftime("%Y%m%dT%H%M%SZ",time.gmtime()))
    diagnostic.mkdir(exist_ok=True)
    (diagnostic/"status.json").write_text(json.dumps({"stage":STAGE,"setup_error":SETUP_ERROR,"run_error":RUN_ERROR,"raw_result_zip":str(RESULT_ZIP) if RESULT_ZIP else None},indent=2))
    for p in WORKSPACE.glob("*.log"):
        shutil.copy2(p,diagnostic/p.name)
    for name in ("setup_failure.txt","run_failure.txt","last_result.json"):
        if (WORKSPACE/name).exists():
            shutil.copy2(WORKSPACE/name,diagnostic/name)
    if RESULT_ZIP is not None and RESULT_ZIP.exists():
        shutil.copy2(RESULT_ZIP,diagnostic/RESULT_ZIP.name)
    RESULT_ZIP=WORKSPACE/"RETURN_Path_A_Plus_Diagnostics.zip"
    with zipfile.ZipFile(RESULT_ZIP,"w",zipfile.ZIP_DEFLATED) as z:
        for p in diagnostic.iterdir():
            z.write(p,p.name)
print("Return this single ZIP:",RESULT_ZIP,flush=True)
display(FileLink(str(RESULT_ZIP)))
try:
    from google.colab import files
    files.download(str(RESULT_ZIP))
except ImportError:
    pass